In [3]:
import pandas as pd
import itertools
import statsmodels.api as sm
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [4]:
def exhaustive_regression(X, y, categorical_target=False):
    """
    Uses:
      - OLS for continuous targets
      - MNLogit for multiclass categorical targets

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix
    y : pd.Series
        Target vector

    Returns
    -------
    results : pd.DataFrame
        DataFrame with feature1, feature2, model type,
        interaction coefficients, and p-values.
    """
    feature_names = X.columns.tolist()

    # Detect target type
    if categorical_target:
        model_type = "multiclass"
        le = LabelEncoder()
        y = le.fit_transform(y)
        classes = le.classes_
    else:
        if pd.api.types.is_numeric_dtype(y):
            model_type = "continuous"
        else:
            model_type = "multiclass"
            le = LabelEncoder()
            y = le.fit_transform(y)
            classes = le.classes_

    results = []

    for f1, f2 in list(itertools.combinations(range(X.shape[1]), 2)):
        # Build design matrix
        feature_pair = X[[feature_names[f1], feature_names[f2]]].copy()
        feature_pair["interaction"] = X[feature_names[f1]] * X[feature_names[f2]]
        feature_pair = sm.add_constant(feature_pair)

        try: 

            if model_type == "continuous":
                # Ordinary Least Squares
                model = sm.OLS(y, feature_pair).fit()
                results.append({
                    "feature1": feature_names[f1],
                    "feature2": feature_names[f2],
                    "model": "OLS",
                    "interaction_coef": model.params.get("interaction", np.nan),
                    "interaction_pval": model.pvalues.get("interaction", np.nan)
                })
            else:
                # Multinomial Logistic Regression
                model = sm.MNLogit(y, feature_pair).fit(disp=False)
                if "interaction" in model.pvalues.index:
                    for class_idx, p in model.pvalues.loc["interaction"].items():
                        results.append({
                            "feature1": feature_names[f1],
                            "feature2": feature_names[f2],
                            "model": "MNLogit",
                            "class": classes[class_idx],
                            "interaction_coef": model.params.loc["interaction", class_idx],
                            "interaction_pval": model.pvalues.loc["interaction", class_idx]
                        })

        except Exception as e:
            results.append({
                "feature1": feature_names[f1],
                "feature2": feature_names[f2],
                "model": model_type,
                "error": str(e)
            })

    return pd.DataFrame(results)

In [5]:
def exhaustive_regression(X, y, categorical_target=False, alpha=0.1):
    """
    Uses:
      - OLS for continuous targets
      - MNLogit for multiclass categorical targets

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix
    y : pd.Series
        Target vector
    categorical_target : bool
        If True, force treating y as categorical.
    alpha : float
        Significance level for flagging after Bonferroni correction.

    Returns
    -------
    results_df : pd.DataFrame
        DataFrame with feature1, feature2, model type,
        interaction coefficients, raw p-values, Bonferroni-adjusted p-values,
        and boolean indicator of significance after correction.
    """
    feature_names = X.columns.tolist()

    # Detect target type
    if categorical_target:
        model_type = "multiclass"
        le = LabelEncoder()
        y_enc = le.fit_transform(y)
        classes = le.classes_
    else:
        if pd.api.types.is_numeric_dtype(y):
            model_type = "continuous"
            y_enc = y.values
        else:
            model_type = "multiclass"
            le = LabelEncoder()
            y_enc = le.fit_transform(y)
            classes = le.classes_

    results = []

    for f1, f2 in itertools.combinations(range(X.shape[1]), 2):
        # Build design matrix
        fname1 = feature_names[f1]
        fname2 = feature_names[f2]
        feature_pair = X[[fname1, fname2]].copy()
        feature_pair["interaction"] = X[fname1] * X[fname2]
        feature_pair = sm.add_constant(feature_pair)

        try:
            if model_type == "continuous":
                # Ordinary Least Squares
                model = sm.OLS(y_enc, feature_pair).fit()
                pval = model.pvalues.get("interaction", np.nan)
                coef = model.params.get("interaction", np.nan)
                results.append({
                    "feature1": fname1,
                    "feature2": fname2,
                    "model": "OLS",
                    "class": None,
                    "interaction_coef": coef,
                    "interaction_pval": pval
                })
            else:
                # Multinomial Logistic Regression
                model = sm.MNLogit(y_enc, feature_pair).fit(disp=False)
                # model.params and model.pvalues are DataFrames indexed by parameter name,
                # columns correspond to classes (0..K-1)
                if "interaction" in model.pvalues.index:
                    # iterate columns (class indices)
                    for class_idx in model.pvalues.columns:
                        pval = model.pvalues.loc["interaction", class_idx]
                        coef = model.params.loc["interaction", class_idx]
                        # map class_idx (int) to actual class label if available
                        class_label = classes[class_idx] if 'classes' in locals() else class_idx
                        results.append({
                            "feature1": fname1,
                            "feature2": fname2,
                            "model": "MNLogit",
                            "class": class_label,
                            "interaction_coef": coef,
                            "interaction_pval": pval
                        })
                else:
                    # No interaction parameter found in results (unexpected), append NaNs
                    results.append({
                        "feature1": fname1,
                        "feature2": fname2,
                        "model": "MNLogit",
                        "class": None,
                        "interaction_coef": np.nan,
                        "interaction_pval": np.nan
                    })

        except Exception as e:
            # keep record of the failure — interaction_pval left as NaN
            results.append({
                "feature1": fname1,
                "feature2": fname2,
                "model": model_type,
                "class": None,
                "interaction_coef": np.nan,
                "interaction_pval": np.nan,
                "error": str(e)
            })

    results_df = pd.DataFrame(results)

    ## bonferonni correction
    
    # Count number of tests performed (non-NaN p-values)
    m = results_df["interaction_pval"].notna().sum()

    # If no tests completed, return early
    if m == 0:
        results_df["interaction_pval_bonf"] = np.nan
        results_df["bonf_significant"] = False
        return results_df

    # Bonferroni adjustment: p_adjusted = min(1, p * m)
    results_df["interaction_pval_bonf"] = results_df["interaction_pval"].apply(
        lambda p: min(1.0, p * m) if pd.notna(p) else np.nan
    )

    # Flag significance after Bonferroni correction
    results_df["bonf_significant"] = results_df["interaction_pval_bonf"].apply(
        lambda p: bool(p < alpha) if pd.notna(p) else False
    )

    # Optional: also include raw alpha threshold flag for convenience
    results_df["raw_significant"] = results_df["interaction_pval"].apply(
        lambda p: bool(p < alpha) if pd.notna(p) else False
    )

    # Add metadata about number of tests and alpha used
    results_df.attrs["n_tests"] = int(m)
    results_df.attrs["alpha"] = float(alpha)

    return results_df

TEST (no perturbation)

In [6]:
pd.set_option('display.max_colwidth', None)

In [8]:
# Load data
df = pd.read_csv("cell_cycle_tidied copy.csv")

# Define features and target
X = df.drop(columns=['phase', 'age', 'PHATE_1', 'PHATE_2'])  # Features
y = df['age']  # Target: age

In [9]:
df = exhaustive_regression(X, y)

In [10]:
n_sig = df["bonf_significant"].sum()
print(f"Number of Bonferroni-significant interactions: {n_sig}")

Number of Bonferroni-significant interactions: 6997


In [11]:
top20 = df.sort_values('interaction_pval_bonf').head(20)
top20

,feature1,feature2,model,class,interaction_coef,interaction_pval,interaction_pval_bonf,bonf_significant,raw_significant
4302,pRB..nuc.median.,p27..nuc.median.,OLS,None,-1.765153,0.000000e+00,0.000000e+00,True,True
6204,pp21..nuc.median.,p21..phospho.total.nuc.,OLS,None,-0.306167,8.255576e-292,2.822664e-287,True,True
5007,p27..nuc.median.,RB..phospho.total.nuc.,OLS,None,-1.552622,5.699296e-290,1.948646e-285,True,True
5029,p27..nuc.median.,ratio,OLS,None,-1.552622,5.699296e-290,1.948646e-285,True,True
6005,pp21..nuc.median.,pp65..nuc.median.,OLS,None,-1.068402,3.311855e-277,1.132356e-272,True,True
3320,RB..nuc.median.,p27..nuc.median.,OLS,None,-1.494585,7.761544e-263,2.653749e-258,True,True
6180,pp21..nuc.median.,ERK..phospho.total.cell.,OLS,None,1.852032,4.006552e-250,1.369880e-245,True,True
4795,p27..nuc.median.,DNA..nuc.median.,OLS,None,-1.535953,2.528862e-240,8.646432e-236,True,True
6202,pp21..nuc.median.,RB..phospho.total.nuc.,OLS,None,-2.926138,1.254877e-222,4.290551e-218,True,True
6224,pp21..nuc.median.,ratio,OLS,None,-2.926138,1.254877e-222,4.290551e-218,True,True


TEST (cancer)

In [13]:
# Load data
df = pd.read_csv("T47D copy.csv")

# Separate features and target
X = df.drop(columns=['Metadata_well'])
X = X.select_dtypes(include=['number'])
y = df['Metadata_well']

In [14]:
df = exhaustive_regression(X, y, categorical_target=True)

In [15]:
n_sig = df["bonf_significant"].sum()
print(f"Number of Bonferroni-significant interactions: {n_sig}")

Number of Bonferroni-significant interactions: 395


In [16]:
top20 = df.sort_values('interaction_pval_bonf').head(20)
top20

,feature1,feature2,model,class,interaction_coef,interaction_pval,interaction_pval_bonf,bonf_significant,raw_significant
759,Intensity_MedianIntensity_pRB,pRB_over_RB,MNLogit,100,0.921525,0.000000e+00,0.000000e+00,True,True
671,Intensity_MedianIntensity_Skp2,Intensity_MedianIntensity_pRB,MNLogit,100,0.912245,0.000000e+00,0.000000e+00,True,True
670,Intensity_MedianIntensity_Skp2,Intensity_MedianIntensity_pRB,MNLogit,10,0.516427,0.000000e+00,0.000000e+00,True,True
695,Intensity_MedianIntensity_cycA,Intensity_MedianIntensity_pRB,MNLogit,100,0.785408,0.000000e+00,0.000000e+00,True,True
643,Intensity_MedianIntensity_RB,Intensity_MedianIntensity_pRB,MNLogit,100,1.124490,0.000000e+00,0.000000e+00,True,True
699,Intensity_MedianIntensity_cycA,pRB_over_RB,MNLogit,100,0.664718,0.000000e+00,0.000000e+00,True,True
395,Intensity_MedianIntensity_Cdh1,pRB_over_RB,MNLogit,100,0.853645,0.000000e+00,0.000000e+00,True,True
715,Intensity_MedianIntensity_cycB1,Intensity_MedianIntensity_pRB,MNLogit,100,0.564088,0.000000e+00,0.000000e+00,True,True
398,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_E2F1,MNLogit,10,0.433850,0.000000e+00,0.000000e+00,True,True
579,Intensity_MedianIntensity_Ki67,pRB_over_RB,MNLogit,100,1.191305,0.000000e+00,0.000000e+00,True,True


In [17]:
df = pd.read_csv("cancer_iloco_decisionTree copy.csv")
significant_count = (df["scores"] - df["ci"] > 0).sum()
print(f"Number of significant interactions (p < 0.1): {significant_count}")

Number of significant interactions (p < 0.1): 47


In [18]:
top_20 = df.nlargest(20, "scores")
top_20

,feature,scores,ci
182,Intensity_MedianIntensity_cycD1 & Intensity_MedianIntensity_pRB,0.007073,0.001302
87,Intensity_MedianIntensity_Cdh1 & Intensity_MedianIntensity_ER,0.005530,0.002360
122,Intensity_MedianIntensity_E2F1 & Intensity_MedianIntensity_pRB,0.003867,0.000771
0,AreaShape_Area & Intensity_IntegratedIntensity_DNA,0.003422,0.000552
106,Intensity_MedianIntensity_Cdt1 & Intensity_MedianIntensity_cycB1,0.003154,0.000785
179,Intensity_MedianIntensity_cycB1 & pRB_over_RB,0.002977,0.000710
178,Intensity_MedianIntensity_cycB1 & Intensity_MedianIntensity_pRB,0.002957,0.000710
35,Intensity_IntegratedIntensity_DNA & Intensity_MedianIntensity_pRB,0.002782,0.000511
184,Intensity_MedianIntensity_cycE & Intensity_MedianIntensity_p21,0.002712,0.000804
18,AreaShape_Area & pRB_over_RB,0.002539,0.000552


DOSE DEPENDANT

In [19]:
# Load data
df = pd.read_csv("T47D copy.csv")

# Separate features and target
X = df.drop(columns=['phase'])
X = X.select_dtypes(include=['number'])
y = df['phase']

In [20]:
df = exhaustive_regression(X, y, categorical_target=True)

/opt/miniconda3/envs/wayne/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:3027: RuntimeWarning: overflow encountered in exp
  eXB = np.column_stack((np.ones(len(X)), np.exp(X)))
/opt/miniconda3/envs/wayne/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:3028: RuntimeWarning: invalid value encountered in divide
  return eXB/eXB.sum(1)[:,None]
/opt/miniconda3/envs/wayne/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:3027: RuntimeWarning: overflow encountered in exp
  eXB = np.column_stack((np.ones(len(X)), np.exp(X)))
/opt/miniconda3/envs/wayne/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:3028: RuntimeWarning: invalid value encountered in divide
  return eXB/eXB.sum(1)[:,None]
/opt/miniconda3/envs/wayne/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:3027: RuntimeWarning: overflow encountered in exp
  eXB = np.column_stack((np.ones(len(X)), np.exp(X)))
/opt/miniconda3/envs/wayne/lib/py

In [21]:
n_sig = df["bonf_significant"].sum()
print(f"Number of Bonferroni-significant interactions: {n_sig}")

Number of Bonferroni-significant interactions: 446


In [22]:
top20 = df.sort_values('interaction_pval_bonf').head(20)
top20

,feature1,feature2,model,class,interaction_coef,interaction_pval,interaction_pval_bonf,bonf_significant,raw_significant
462,Intensity_MedianIntensity_Ki67,Metadata_well,MNLogit,G0,0.007942,0.000000e+00,0.000000e+00,True,True
1,AreaShape_Area,Intensity_IntegratedIntensity_DNA,MNLogit,G1,-1.149329,0.000000e+00,0.000000e+00,True,True
2,AreaShape_Area,Intensity_IntegratedIntensity_DNA,MNLogit,G2/M,-0.967918,0.000000e+00,0.000000e+00,True,True
439,Intensity_MedianIntensity_Ki67,Intensity_MedianIntensity_Skp2,MNLogit,G1,-0.473036,0.000000e+00,0.000000e+00,True,True
315,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_E2F1,MNLogit,G0,-0.407602,0.000000e+00,0.000000e+00,True,True
321,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_Ki67,MNLogit,G0,-0.794763,0.000000e+00,0.000000e+00,True,True
322,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_Ki67,MNLogit,G1,-1.073437,0.000000e+00,0.000000e+00,True,True
440,Intensity_MedianIntensity_Ki67,Intensity_MedianIntensity_Skp2,MNLogit,G2/M,-0.681807,0.000000e+00,0.000000e+00,True,True
327,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_RB,MNLogit,G0,-0.404397,0.000000e+00,0.000000e+00,True,True
330,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_Skp2,MNLogit,G0,-0.930711,0.000000e+00,0.000000e+00,True,True


In [23]:
df = pd.read_csv("cancer_iloco_ensemble copy.csv")
significant_count = (df["scores"] - df["ci"] > 0).sum()
print(f"Number of significant interactions (p < 0.1): {significant_count}")

Number of significant interactions (p < 0.1): 82


In [24]:
top_20 = df.nlargest(20, "scores")
top_20

,feature,scores,ci
209,pRB_over_RB & Metadata_well,0.048161,0.002420
38,Intensity_IntegratedIntensity_DNA & Metadata_well,0.026614,0.002364
208,Intensity_MedianIntensity_pRB & Metadata_well,0.014119,0.000568
171,Intensity_MedianIntensity_RB & Intensity_MedianIntensity_pRB,0.010555,0.000386
173,Intensity_MedianIntensity_RB & Metadata_well,0.003133,0.000386
107,Intensity_MedianIntensity_Cdt1 & Intensity_MedianIntensity_Ki67,0.002143,0.000365
163,Intensity_MedianIntensity_PR & pRB_over_RB,0.002139,0.000217
144,Intensity_MedianIntensity_Ki67 & Intensity_MedianIntensity_PR,0.002053,0.000456
103,Intensity_MedianIntensity_Cdh1 & pRB_over_RB,0.001955,0.000385
193,Intensity_MedianIntensity_cycB1 & pRB_over_RB,0.001893,0.000545


In [27]:
import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2
import warnings

def _zscore_and_clip(s, clip_sds=8.0, eps=1e-12):
    m = float(np.nanmean(s))
    v = float(np.nanvar(s))
    sd = np.sqrt(v) if v > 0 else 1.0
    z = (s - m) / (sd + eps)
    # gentle clip to avoid extreme logits; keeps LR invariant up to reparam
    z = np.clip(z, -clip_sds, clip_sds)
    return z

def _fit_test_continuous(y, X_full):
    X_red = X_full.drop(columns=["interaction"])
    full = sm.OLS(y, X_full).fit()
    red  = sm.OLS(y, X_red).fit()
    F, pval, _df = full.compare_f_test(red)
    coef = full.params.get("interaction", np.nan)
    return float(coef), float(pval), int(full.nobs)

def _fit_test_binary(y, X_full):
    X_red = X_full.drop(columns=["interaction"])
    full = sm.Logit(y, X_full).fit(disp=False)
    red  = sm.Logit(y, X_red).fit(disp=False)
    lr = 2.0 * (full.llf - red.llf)
    pval = chi2.sf(lr, df=1)
    coef = full.params.get("interaction", np.nan)
    return float(coef), float(pval), int(full.nobs)

def _fit_test_multiclass(y, X_full):
    X_red = X_full.drop(columns=["interaction"])
    # treat RuntimeWarnings as errors so we can catch overflow cleanly
    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=RuntimeWarning)
        full = sm.MNLogit(y, X_full).fit(disp=False)
        red  = sm.MNLogit(y, X_red).fit(disp=False)
    lr = 2.0 * (full.llf - red.llf)
    k = full.params.shape[1]  # non-baseline classes
    pval = chi2.sf(lr, df=k)
    try:
        beta = full.params.loc["interaction"].to_numpy()
        effect = float(np.linalg.norm(beta))
    except Exception:
        effect = np.nan
    return effect, float(pval), int(full.nobs)

def exhaustive_interactions_joint(
    X, y, categorical_target=False, alpha=0.05, stabilize_interaction=True,
    min_per_class=3
):
    """
    One joint test per (feature1, feature2):
      - continuous y: OLS nested F
      - binary y: Logit LR (df=1)
      - multiclass y: MNLogit joint LR across non-baseline classes (df=K-1)

    Bonferroni is applied across PAIRS.
    """
    X = X.copy()
    y = pd.Series(y).copy()

    mask = X.notna().all(axis=1) & y.notna()
    X, y = X.loc[mask], y.loc[mask]

    if not categorical_target and pd.api.types.is_numeric_dtype(y):
        target_kind = "continuous"
        y_vec = y.to_numpy(dtype=float)
    else:
        le = LabelEncoder()
        y_enc = le.fit_transform(y.to_numpy())
        target_kind = "binary" if np.unique(y_enc).size == 2 else "multiclass"
        y_vec = y_enc

    feats = X.columns.tolist()
    rows = []
    for i, j in itertools.combinations(range(X.shape[1]), 2):
        n1, n2 = feats[i], feats[j]
        Xi = pd.DataFrame({
            n1: X.iloc[:, i].to_numpy(),
            n2: X.iloc[:, j].to_numpy()
        }, index=X.index)

        # build interaction and stabilize only this column (prevents softmax overflow)
        inter = Xi[n1] * Xi[n2]
        if stabilize_interaction:
            inter = _zscore_and_clip(inter)
        Xi["interaction"] = inter
        Xi = sm.add_constant(Xi, has_constant="add")

        # quick guard for tiny classes (common cause of separation/overflow)
        if target_kind in ("binary", "multiclass"):
            vc = pd.Series(y_vec).value_counts()
            if (vc.min() < min_per_class) or (vc.size < (2 if target_kind=="binary" else 3)):
                rows.append({
                    "feature1": n1, "feature2": n2, "pair": f"{n1}×{n2}",
                    "model": "MNLogit-Joint" if target_kind=="multiclass" else "Logit",
                    "interaction_effect": np.nan, "pval": np.nan, "n": len(y_vec),
                    "error": "too_few_samples_in_a_class"
                })
                continue

        try:
            if target_kind == "continuous":
                effect, pval, nobs = _fit_test_continuous(y_vec, Xi)
                model = "OLS"
            elif target_kind == "binary":
                effect, pval, nobs = _fit_test_binary(y_vec, Xi)
                model = "Logit"
            else:
                effect, pval, nobs = _fit_test_multiclass(y_vec, Xi)
                model = "MNLogit-Joint"
            rows.append({
                "feature1": n1, "feature2": n2, "pair": f"{n1}×{n2}",
                "model": model, "interaction_effect": effect, "pval": pval, "n": nobs
            })
        except Exception as e:
            rows.append({
                "feature1": n1, "feature2": n2, "pair": f"{n1}×{n2}",
                "model": "MNLogit-Joint" if target_kind=="multiclass" else ("Logit" if target_kind=="binary" else "OLS"),
                "interaction_effect": np.nan, "pval": np.nan, "n": len(y_vec),
                "error": str(e)
            })

    out = pd.DataFrame(rows)
    m = out["pval"].notna().sum()
    if m > 0:
        out["pval_bonf"] = out["pval"].apply(lambda p: min(1.0, p*m) if pd.notna(p) else np.nan)
        out["bonf_significant"] = out["pval_bonf"] < alpha
    else:
        out["pval_bonf"] = np.nan
        out["bonf_significant"] = False

    out.attrs["n_tests"] = int(m)
    out.attrs["alpha"] = float(alpha)
    return out

# === How to run ===
res = exhaustive_interactions_joint(X, y, categorical_target=True, alpha=0.05)
print("Pairs tested:", res["pval"].notna().sum())
print("Bonferroni-significant pairs:", int(res["bonf_significant"].sum()))
res.loc[res["bonf_significant"]].sort_values("pval_bonf").head(20)

Pairs tested: 189
Bonferroni-significant pairs: 187


,feature1,feature2,pair,model,interaction_effect,pval,n,error,pval_bonf,bonf_significant
0,AreaShape_Area,Intensity_IntegratedIntensity_DNA,AreaShape_Area×Intensity_IntegratedIntensity_DNA,MNLogit-Joint,3.056914,0.000000e+00,64502,NaN,0.000000e+00,True
132,Intensity_MedianIntensity_ER,Intensity_MedianIntensity_Ki67,Intensity_MedianIntensity_ER×Intensity_MedianIntensity_Ki67,MNLogit-Joint,1.471373,0.000000e+00,64502,NaN,0.000000e+00,True
112,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_cycB1,Intensity_MedianIntensity_Cdt1×Intensity_MedianIntensity_cycB1,MNLogit-Joint,1.578410,0.000000e+00,64502,NaN,0.000000e+00,True
36,Intensity_IntegratedIntensity_DNA,Intensity_MedianIntensity_pRB,Intensity_IntegratedIntensity_DNA×Intensity_MedianIntensity_pRB,MNLogit-Joint,1.051426,0.000000e+00,64502,NaN,0.000000e+00,True
146,Intensity_MedianIntensity_Ki67,Intensity_MedianIntensity_Skp2,Intensity_MedianIntensity_Ki67×Intensity_MedianIntensity_Skp2,MNLogit-Joint,2.970572,0.000000e+00,64502,NaN,0.000000e+00,True
111,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_cycA,Intensity_MedianIntensity_Cdt1×Intensity_MedianIntensity_cycA,MNLogit-Joint,2.154959,0.000000e+00,64502,NaN,0.000000e+00,True
151,Intensity_MedianIntensity_Ki67,Intensity_MedianIntensity_p21,Intensity_MedianIntensity_Ki67×Intensity_MedianIntensity_p21,MNLogit-Joint,1.204351,0.000000e+00,64502,NaN,0.000000e+00,True
26,Intensity_IntegratedIntensity_DNA,Intensity_MedianIntensity_ER,Intensity_IntegratedIntensity_DNA×Intensity_MedianIntensity_ER,MNLogit-Joint,0.874059,0.000000e+00,64502,NaN,0.000000e+00,True
110,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_Skp2,Intensity_MedianIntensity_Cdt1×Intensity_MedianIntensity_Skp2,MNLogit-Joint,1.878662,0.000000e+00,64502,NaN,0.000000e+00,True
109,Intensity_MedianIntensity_Cdt1,Intensity_MedianIntensity_RB,Intensity_MedianIntensity_Cdt1×Intensity_MedianIntensity_RB,MNLogit-Joint,1.827585,0.000000e+00,64502,NaN,0.000000e+00,True
